In [11]:
import geopandas as gpd
import ee
ee.Authenticate()
ee.Initialize()

# import fiona
# fiona.listlayers("https://tubvsig-so2sat-vm1.srv.mwn.de/geoserver/ows?service=WFS&request=GetCapabilities")

In [12]:
# -------------------------------
# USER REGION
# -------------------------------

min_lon, min_lat = -121.74768646582278, 39.51545530875191
max_lon, max_lat = -121.38224589615993, 39.980592378624905

region = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

# Build dynamic description name
desc = f"Building_Height_{min_lon}_{max_lon}_{min_lat}_{max_lat}".replace("-", "w").replace(".", "")
print("Export description:", desc)

# -------------------------------
# LOAD TILE
# -------------------------------
tile = ee.FeatureCollection(
    "projects/sat-io/open-datasets/GLOBAL_BUILDING_ATLAS/w125_n40_w120_n35"
)

# Clip buildings to region
clipped = tile.filterBounds(region).map(
    lambda f: f.intersection(region, maxError=1)
)

count = clipped.size().getInfo()
print("Buildings in region:", count)

# -------------------------------
# RASTERIZE HEIGHT FIELD
# -------------------------------
# Use "height" column and rasterize at 30 m resolution
height_raster = clipped.reduceToImage(
    properties=["height"],
    reducer=ee.Reducer.first()
).rename("building_height")

# -------------------------------
# EXPORT AS GEOtTIFF
# -------------------------------
task = ee.batch.Export.image.toDrive(
    image=height_raster,
    description=desc,
    folder=None,                # optional
    fileNamePrefix=desc,
    region=region,
    scale=30,
    crs="EPSG:4326",
    fileFormat="GeoTIFF",
)

task.start()

print("TIFF export started. Check Tasks tab.")


Export description: Building_Height_w12174768646582278_w12138224589615993_3951545530875191_39980592378624905
Buildings in region: 23018
TIFF export started. Check Tasks tab.


In [ ]:
# Script to download land cover

import ee
ee.Initialize()

# ---------------------------
# 1) Define your region of interest
# ---------------------------
min_lon, min_lat = -115.1, 34.9
max_lon, max_lat = -115, 35

region = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])
desc = f"LC_{min_lon}_{max_lon}_{min_lat}_{max_lat}".replace("-", "w").replace(".", "")

# ---------------------------
# 2) Load ESA WorldCover (2020) and clip to region
#    Dataset: "ESA/WorldCover/v100"
# ---------------------------
lc_clip = (ee.ImageCollection("ESA/WorldCover/v100").mosaic().select('Map').clip(region))

# ---------------------------
# 3) Set export parameters
# ---------------------------
task = ee.batch.Export.image.toDrive(
    image = lc_clip,
    description = "ESA_WorldCover_subset",
    folder = "EarthEngine",
    fileNamePrefix = desc,
    scale = 10,  # 10 m resolution for WorldCover
    region = region,
    fileFormat = "GeoTIFF"
)

# ---------------------------
# 5) Start the export
# ---------------------------
task.start()
print("Export started. Check your Google Drive.")

Export started. Check your Google Drive.


In [7]:
import ee
ee.Initialize()

# --------------------------------------
# Define Region rectangle (same as GEE)
# --------------------------------------
min_lon, min_lat = -115.1, 34.9
max_lon, max_lat = -115, 35
region = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

desc = f"LAI_{min_lon}_{max_lon}_{min_lat}_{max_lat}".replace("-", "w").replace(".", "")
# --------------------------------------
# LAI asset information
# --------------------------------------
ASSET_ROOT = 'projects/tc-global-urban/assets/'

FILENAMES = [
  'LAI_Grid_30deg_101_2020-07-02',
  'LAI_Grid_30deg_102_2020-07-02',
  'LAI_Grid_30deg_103_2020-07-02',
  'LAI_Grid_30deg_104_2020-07-02',
  'LAI_Grid_30deg_105_2020-07-02',
  'LAI_Grid_30deg_107_2020-07-02',
  'LAI_Grid_30deg_108_2020-07-02',
  'LAI_Grid_30deg_0_2020-07-02',
  'LAI_Grid_30deg_1_2020-07-02',
  'LAI_Grid_30deg_2_2020-07-02',
  'LAI_Grid_30deg_3_2020-07-02',
  'LAI_Grid_30deg_4_2020-07-02',
  'LAI_Grid_30deg_5_2020-07-02',
  'LAI_Grid_30deg_6_2020-07-02',
  'LAI_Grid_30deg_7_2020-07-02',
  'LAI_Grid_30deg_8_2020-07-02',
  'LAI_Grid_30deg_9_2020-07-02',
  'LAI_Grid_30deg_10_2020-07-02',  
  'LAI_Grid_30deg_11NE_2020-07-02',
  'LAI_Grid_30deg_11NW_2020-07-02',
  'LAI_Grid_30deg_11SE_2020-07-02',
  'LAI_Grid_30deg_11SW_2020-07-02',
  'LAI_Grid_30deg_12_2020-07-02', 
  'LAI_Grid_30deg_13_2020-07-02',
  'LAI_Grid_30deg_14_2020-07-02', 
  'LAI_Grid_30deg_15_2020-07-02',
  'LAI_Grid_30deg_16NE_2020-07-02',
  'LAI_Grid_30deg_16NW_2020-07-02',
  'LAI_Grid_30deg_16SE_2020-07-02',
  'LAI_Grid_30deg_16SW_2020-07-02',
  'LAI_Grid_30deg_17_2020-07-02',
  'LAI_Grid_30deg_18_2020-07-02', 
  'LAI_Grid_30deg_19_2020-07-02',
  'LAI_Grid_30deg_20_2020-07-02',  
  'LAI_Grid_30deg_21NE_2020-07-02',
  'LAI_Grid_30deg_21NW_2020-07-02',
  'LAI_Grid_30deg_21SE_2020-07-02',
  'LAI_Grid_30deg_21SW_2020-07-02',
  'LAI_Grid_30deg_22_2020-07-02',
  'LAI_Grid_30deg_23_2020-07-02',
  'LAI_Grid_30deg_24_2020-07-02', 
  'LAI_Grid_30deg_25_2020-07-02',
  'LAI_Grid_30deg_26_2020-07-02',
  'LAI_Grid_30deg_27_2020-07-02',
  'LAI_Grid_30deg_28_2020-07-02',
  'LAI_Grid_30deg_29_2020-07-02',
  'LAI_Grid_30deg_30_2020-07-02',
  'LAI_Grid_30deg_31_2020-07-02',
  'LAI_Grid_30deg_32_2020-07-02',
  'LAI_Grid_30deg_33_2020-07-02',
  'LAI_Grid_30deg_34_2020-07-02',
  'LAI_Grid_30deg_35_2020-07-02',
  'LAI_Grid_30deg_36_2020-07-02',
  'LAI_Grid_30deg_37_2020-07-02',
  'LAI_Grid_30deg_38_2020-07-02',
  'LAI_Grid_30deg_39_2020-07-02',
  'LAI_Grid_30deg_40_2020-07-02',
  'LAI_Grid_30deg_41_2020-07-02',
  'LAI_Grid_30deg_42_2020-07-02',
  'LAI_Grid_30deg_43_2020-07-02',
  'LAI_Grid_30deg_44_2020-07-02',
  'LAI_Grid_30deg_45_2020-07-02',
  'LAI_Grid_30deg_46_2020-07-02',
  'LAI_Grid_30deg_47_2020-07-02',
  'LAI_Grid_30deg_48_2020-07-02',
  'LAI_Grid_30deg_49_2020-07-02',
  'LAI_Grid_30deg_50_2020-07-02',
  'LAI_Grid_30deg_51_2020-07-02',
  'LAI_Grid_30deg_52_2020-07-02',
  'LAI_Grid_30deg_53_2020-07-02',
  'LAI_Grid_30deg_54_2020-07-02',
  'LAI_Grid_30deg_55_2020-07-02',
  'LAI_Grid_30deg_56_2020-07-02',
  'LAI_Grid_30deg_101_2020-07-02',
  'LAI_Grid_30deg_102_2020-07-02',
  'LAI_Grid_30deg_103_2020-07-02',
  'LAI_Grid_30deg_104_2020-07-02',
  'LAI_Grid_30deg_105_2020-07-02',
  'LAI_Grid_30deg_107_2020-07-02',
  'LAI_Grid_30deg_108_2020-07-02'
];

# --------------------------------------
# Function to load + tag each LAI tile
# --------------------------------------
def load_lai_tile(name):
    img = ee.Image(ASSET_ROOT + name).select([0]).rename('LAI')

    # parse index + quadrant (same regex as JS)
    import re
    m = re.search(r'30deg_(\d+)([A-Z]{0,2})_', name)
    idx = int(m.group(1)) if m else None
    quad = m.group(2) if (m and m.group(2)) else ''

    return img.set({
        'grid_idx': idx,
        'quadrant': quad,
        'system:index': name
    })

# --------------------------------------
# Build ImageCollection
# --------------------------------------
lai_images = [load_lai_tile(name) for name in FILENAMES]
lai_IC = ee.ImageCollection(lai_images).sort('system:index')

# Mosaic + clip
lai_mosaic = lai_IC.mosaic().clip(region)

# --------------------------------------
# Print to verify
# --------------------------------------
print("LAI Mosaic:", lai_mosaic.getInfo())

#export to tiff
task_lai = ee.batch.Export.image.toDrive(
    image = lai_mosaic,
    description = "LAI_Mosaic_subset",
    folder = "EarthEngine",
    fileNamePrefix = desc,
    scale = 30,  # 30 m resolution for LAI
    region = region,
    fileFormat = "GeoTIFF"
)

task_lai.start()
print("LAI export started. Check Tasks tab.")

LAI Mosaic: {'type': 'Image', 'bands': [{'id': 'LAI', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -2147483648, 'max': 2147483647}, 'dimensions': [2, 2], 'origin': [-116, 34], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:footprint': {'type': 'Polygon', 'coordinates': [[[-115.1, 34.9], [-115, 34.9], [-115, 35], [-115.1, 35], [-115.1, 34.9]]]}}}
LAI export started. Check Tasks tab.
